# Consolidate Legalo Model Versions into One Kaggle Dataset Version

Bu notebook, `ardayildiz29/legal-models-tr-finetuned` dataset'indeki eski version'ları indirip tek bir klasör yapısında birleştirir:

```text
legalo_models_merged/
├── bgem3/          # dataset version 1
├── reranker_ft/    # dataset version 2
├── qwen_qlora_ft/  # dataset version 3
├── manifest.json
└── dataset-metadata.json
```

Önceki notebook'taki ana hata şuydu: Kaggle input klasörü çoğu zaman sadece latest version'ı gösterir; `version 1`, `version 2`, `version 3` otomatik ayrı klasörler olarak gelmez. Bu notebook eski versiyonları `kagglehub.dataset_download(..., version=...)` ile indirir.


In [8]:
from pathlib import Path
import os
import sys
import json
import shutil
import hashlib
import subprocess
from datetime import datetime, timezone

DATASET_HANDLE = "ardayildiz29/legal-models-tr-finetuned"

# Hangi Kaggle dataset version'ı hangi final klasöre gidecek?
VERSION_SPECS = {
    "bgem3": {
        "version": 1,
        "description": "BGE-M3 embedding model",
        "subfolder": None,       # version 1 root'ta doğrudan dosyalar var
    },
    "reranker_ft": {
        "version": 2,
        "description": "Fine-tuned reranker model",
        "subfolder": "reranker_ft",   # version 2 içinde reranker_ft/ alt klasörü var
    },
    "qwen_qlora_ft": {
        "version": 2,
        "description": "Qwen QLoRA fine-tuned model",
        "subfolder": "qwen_qlora_ft", # version 2 içinde qwen_qlora_ft/ alt klasörü var
    },
}

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DOWNLOAD_DIR = WORK_DIR / "downloaded_versions"
OUT_DIR = WORK_DIR / "legalo_models_merged"
ZIP_BASE = WORK_DIR / "legalo_models_merged"

print("WORK_DIR:", WORK_DIR)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)
print("OUT_DIR:", OUT_DIR)


WORK_DIR: /kaggle/working
DOWNLOAD_DIR: /kaggle/working/downloaded_versions
OUT_DIR: /kaggle/working/legalo_models_merged


## 1) KaggleHub kurulumu

Kaggle Notebook içinde çoğu zaman `kagglehub` hazır gelir. Hazır değilse aşağıdaki hücre otomatik kurar.


In [9]:
try:
    import kagglehub
    print("kagglehub zaten kurulu.")
except ImportError:
    print("kagglehub bulunamadı, kuruluyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub
    print("kagglehub kuruldu.")


kagglehub zaten kurulu.


## 2) Dataset version'larını indir

Bu hücre version 1, 2 ve 3'ü ayrı ayrı indirir. Eğer local/Colab ortamında çalışıyorsan Kaggle credential gerekebilir. Kaggle Notebook içinde genelde sorunsuz çalışır.


In [10]:
def safe_rmtree(path: Path):
    if path.exists():
        shutil.rmtree(path)

safe_rmtree(DOWNLOAD_DIR)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATHS = {}

for final_folder, spec in VERSION_SPECS.items():
    version = spec["version"]
    subfolder = spec.get("subfolder")
    print(f"\nDownloading {DATASET_HANDLE} version {version} for {final_folder}...")
    downloaded_path = kagglehub.dataset_download(f"{DATASET_HANDLE}/versions/{version}")
    downloaded_path = Path(downloaded_path)
    print("Downloaded/cache path:", downloaded_path)

    # Eğer version içinde alt klasör varsa (P2'nin staging yapısı), doğru alt klasörü seç
    if subfolder is not None:
        src_path = downloaded_path / subfolder
        if not src_path.exists():
            raise FileNotFoundError(f"Alt klasör bulunamadı: {src_path}")
        print(f"Alt klasör seçildi: {subfolder}/")
    else:
        src_path = downloaded_path

    # KaggleHub cache read-only olabilir, working'e kopyala
    local_copy = DOWNLOAD_DIR / f"v{version}_{final_folder}"
    if local_copy.exists():
        shutil.rmtree(local_copy)
    shutil.copytree(src_path, local_copy)
    SOURCE_PATHS[final_folder] = local_copy
    print("Local copy:", local_copy)

print("\nSOURCE_PATHS:")
for k, v in SOURCE_PATHS.items():
    print(f"{k:14s} -> {v}")



Downloaded/cache path: /kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned
Local copy: /kaggle/working/downloaded_versions/v1_bgem3

Downloaded/cache path: /kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned
Alt klasör seçildi: reranker_ft/
Local copy: /kaggle/working/downloaded_versions/v2_reranker_ft

Downloaded/cache path: /kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned
Alt klasör seçildi: qwen_qlora_ft/
Local copy: /kaggle/working/downloaded_versions/v2_qwen_qlora_ft

SOURCE_PATHS:
bgem3          -> /kaggle/working/downloaded_versions/v1_bgem3
reranker_ft    -> /kaggle/working/downloaded_versions/v2_reranker_ft
qwen_qlora_ft  -> /kaggle/working/downloaded_versions/v2_qwen_qlora_ft


## 3) İndirilen klasörleri kontrol et

Burada her version'ın içinde gerçekten dosya var mı diye bakıyoruz.


In [11]:
def tree_preview(path: Path, max_items: int = 60):
    path = Path(path)
    print(f"\n--- {path} ---")
    if not path.exists():
        print("YOK")
        return
    items = sorted(path.rglob("*"))
    if not items:
        print("BOŞ")
        return
    for i, p in enumerate(items[:max_items], 1):
        rel = p.relative_to(path)
        suffix = "/" if p.is_dir() else ""
        print(f"{i:03d}. {rel}{suffix}")
    if len(items) > max_items:
        print(f"... toplam {len(items)} item var, ilk {max_items} gösterildi.")

missing = []
empty = []
for target, src in SOURCE_PATHS.items():
    src = Path(src)
    if not src.exists():
        missing.append(target)
    elif not any(src.iterdir()):
        empty.append(target)
    else:
        tree_preview(src, max_items=40)

if missing or empty:
    raise RuntimeError(f"Kaynak problemi var. missing={missing}, empty={empty}")
else:
    print("\nTüm kaynak klasörler bulundu ve boş değil.")



--- /kaggle/working/downloaded_versions/v1_bgem3 ---
001. 1_Pooling/
002. 1_Pooling/config.json
003. README.md
004. config.json
005. config_sentence_transformers.json
006. model.safetensors
007. modules.json
008. sentence_bert_config.json
009. sentencepiece.bpe.model
010. special_tokens_map.json
011. tokenizer.json
012. tokenizer_config.json

--- /kaggle/working/downloaded_versions/v2_reranker_ft ---
001. config.json
002. model.safetensors
003. special_tokens_map.json
004. tokenizer.json
005. tokenizer_config.json
006. vocab.txt

--- /kaggle/working/downloaded_versions/v2_qwen_qlora_ft ---
001. README.md
002. adapter_config.json
003. adapter_model.safetensors
004. added_tokens.json
005. chat_template.jinja
006. merges.txt
007. special_tokens_map.json
008. tokenizer.json
009. tokenizer_config.json
010. vocab.json

Tüm kaynak klasörler bulundu ve boş değil.


## 4) Tek klasör yapısına kopyala

Her version'ın tüm içeriği finalde kendi model klasörüne kopyalanır.


In [12]:
safe_rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def copy_contents(src: Path, dst: Path):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, symlinks=False)
        else:
            shutil.copy2(item, target)

for final_folder, src in SOURCE_PATHS.items():
    dst = OUT_DIR / final_folder
    copy_contents(Path(src), dst)
    print(f"Copied: {src} -> {dst}")

print("\nFinal klasör yapısı:")
tree_preview(OUT_DIR, max_items=160)


Copied: /kaggle/working/downloaded_versions/v1_bgem3 -> /kaggle/working/legalo_models_merged/bgem3
Copied: /kaggle/working/downloaded_versions/v2_reranker_ft -> /kaggle/working/legalo_models_merged/reranker_ft
Copied: /kaggle/working/downloaded_versions/v2_qwen_qlora_ft -> /kaggle/working/legalo_models_merged/qwen_qlora_ft

Final klasör yapısı:

--- /kaggle/working/legalo_models_merged ---
001. bgem3/
002. bgem3/1_Pooling/
003. bgem3/1_Pooling/config.json
004. bgem3/README.md
005. bgem3/config.json
006. bgem3/config_sentence_transformers.json
007. bgem3/model.safetensors
008. bgem3/modules.json
009. bgem3/sentence_bert_config.json
010. bgem3/sentencepiece.bpe.model
011. bgem3/special_tokens_map.json
012. bgem3/tokenizer.json
013. bgem3/tokenizer_config.json
014. qwen_qlora_ft/
015. qwen_qlora_ft/README.md
016. qwen_qlora_ft/adapter_config.json
017. qwen_qlora_ft/adapter_model.safetensors
018. qwen_qlora_ft/added_tokens.json
019. qwen_qlora_ft/chat_template.jinja
020. qwen_qlora_ft/merg

## 5) Manifest ve metadata oluştur

`manifest.json` dosyası hangi klasörün hangi version'dan geldiğini ve dosya özetlerini tutar.


In [13]:
def file_sha256(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "dataset_handle": DATASET_HANDLE,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "layout": {},
    "files": [],
}

for final_folder, spec in VERSION_SPECS.items():
    manifest["layout"][final_folder] = {
        "source_dataset_version": spec["version"],
        "description": spec["description"],
        "path": f"{final_folder}/",
    }

for file in sorted(OUT_DIR.rglob("*")):
    if file.is_file():
        rel = file.relative_to(OUT_DIR).as_posix()
        # Büyük model dosyaları için sha256 zaman alabilir. Yine de bütünlük için hesaplıyoruz.
        manifest["files"].append({
            "path": rel,
            "size_bytes": file.stat().st_size,
            "sha256": file_sha256(file),
        })

with open(OUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

metadata = {
    "title": "Legal Models TR Finetuned",
    "id": DATASET_HANDLE,
    "licenses": [{"name": "CC0-1.0"}],
}

with open(OUT_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("manifest.json ve dataset-metadata.json oluşturuldu.")
print("Toplam dosya sayısı:", len(manifest["files"]))
print("Toplam boyut GB:", round(sum(x["size_bytes"] for x in manifest["files"]) / (1024**3), 3))


manifest.json ve dataset-metadata.json oluşturuldu.
Toplam dosya sayısı: 27
Toplam boyut GB: 2.313


## 6) ZIP oluştur

Kaggle output bölümünden indirebilmen için zip de oluşturulur.


In [14]:
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", OUT_DIR)
print("ZIP oluşturuldu:", zip_path)


ZIP oluşturuldu: /kaggle/working/legalo_models_merged.zip


## 7) Kaggle dataset'e yeni version olarak yükle

Aşağıdaki hücre varsayılan olarak upload yapmaz. Kontrol ettikten sonra `UPLOAD_TO_KAGGLE = True` yapıp çalıştır.

> Bu işlem `ardayildiz29/legal-models-tr-finetuned` dataset sayfasına yeni bir version ekler.


## 8) RAG içinde kullanım

Yeni version yayınlandıktan sonra model path'lerini şöyle kullanabilirsin:

```python
from pathlib import Path

MODEL_ROOT = Path("/kaggle/input/legal-models-tr-finetuned")

BGEM3_PATH = MODEL_ROOT / "bgem3"
RERANKER_PATH = MODEL_ROOT / "reranker_ft"
QWEN_PATH = MODEL_ROOT / "qwen_qlora_ft"
```


In [15]:
UPLOAD_TO_KAGGLE = True

print("Upload klasörü:", OUT_DIR)
print("dataset-metadata.json var mı?", (OUT_DIR / "dataset-metadata.json").exists())
print("manifest.json var mı?", (OUT_DIR / "manifest.json").exists())

if UPLOAD_TO_KAGGLE:
    cmd = [
        "kaggle", "datasets", "version",
        "-p", str(OUT_DIR),
        "-m", "v2: bgem3 (MNRL+seq512), reranker_ft (pos_weight fix), qwen_qlora_ft (max_grad+5000)",
        "--dir-mode", "zip",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("UPLOAD_TO_KAGGLE=False. Upload için True yapıp bu hücreyi tekrar çalıştır.")


Upload klasörü: /kaggle/working/legalo_models_merged
dataset-metadata.json var mı? True
manifest.json var mı? True
Running: kaggle datasets version -p /kaggle/working/legalo_models_merged -m v2: bgem3 (MNRL+seq512), reranker_ft (pos_weight fix), qwen_qlora_ft (max_grad+5000) --dir-mode zip
Starting upload for file qwen_qlora_ft.zip


100%|██████████| 34.1M/34.1M [00:00<00:00, 49.8MB/s]


Upload successful: qwen_qlora_ft.zip (34MB)
Starting upload for file reranker_ft.zip


100%|██████████| 117M/117M [00:01<00:00, 115MB/s]  
  0%|          | 0.00/5.00k [00:00<?, ?B/s]

Upload successful: reranker_ft.zip (117MB)
Starting upload for file manifest.json


100%|██████████| 5.00k/5.00k [00:00<00:00, 13.5kB/s]


Upload successful: manifest.json (5KB)
Starting upload for file bgem3.zip


100%|██████████| 1.67G/1.67G [00:12<00:00, 145MB/s] 


Upload successful: bgem3.zip (2GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ardayildiz29/legal-models-tr-finetuned
